[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

### My notes:
#### 1. RoPE Rotation:

For each pair of feature dimensions in one Q or K vector:

`[x_even, x_odd]` rotate the pair by angle `θ`:

$$
\begin{bmatrix}
x'_{\text{even}} \\
x'_{\text{odd}}
\end{bmatrix}
=
\begin{bmatrix}
\cos\theta & -\sin\theta \\
\sin\theta & \cos\theta
\end{bmatrix}
\begin{bmatrix}
x_{\text{even}} \\
x_{\text{odd}}
\end{bmatrix}
$$

Examples of the pairs:

`(dim 0, dim 1), (dim 2, dim 3), (dim 4, dim 5), ...`

Important:

- `x_even` and `x_odd` are two feature dimensions from the **same token vector**
- `θ` depends on the token position and the dimension-pair index
- The same rotation is applied separately to both `Q` and `K`

#### 2. Why RoPE Gives Relative Position?

Attention compares the **query at token position `i`** with the **key at token position `j`**:

`q_i · k_j`

RoPE rotates them using their own token positions:

`q_i → R(θ_i)q_i`
`k_j → R(θ_j)k_j`

So:

$$
(R(\theta_i)q_i)^T(R(\theta_j)k_j)
=q_i^T R(\theta_j-\theta_i)k_j
$$

The important part is the angle difference:

$$
\theta_j-\theta_i
$$

For each adjacent pair of feature values along the feature dimension, RoPE uses a fixed rotation frequency `ω`:

* `(x_0, x_1)` uses `ω_0`
* `(x_2, x_3)` uses `ω_1`
* `(x_4, x_5)` uses `ω_2`
* ...

For one fixed feature pair:

$$
\omega = \frac{1}{10000^{2p/D}}
$$

where `p` is the feature-pair index.

The rotation angle at token position `pos` is:

$$
\theta_{pos} = pos \cdot \omega
$$

Therefore, for query position `i` and key position `j`:

$$
\theta_i = i\omega
, \theta_j = j\omega
$$

so:

$$
\theta_j-\theta_i
= j\omega-i\omega
=(j-i)\omega
$$

Thus:

$$
\theta_j-\theta_i \propto j-i
$$

So the **positional effect** in `q_i · k_j` depends on the relative token distance `j - i`, not on the absolute positions separately.


#### 3. Why group features into pairs?

RoPE needs pairs because a rotation is a **2D operation**.

It groups adjacent feature dimensions:

`(x_0, x_1), (x_2, x_3), ...`

and treats each pair as one 2D vector:

$$
\begin{bmatrix}
x_{\text{even}} \\
x_{\text{odd}}
\end{bmatrix}
$$

Each pair gets one frequency `ω` and is rotated by:

$$
\theta = pos \cdot \omega
$$

A single scalar feature cannot be continuously rotated by an angle, so RoPE uses **two dimensions per rotation plane**.

Therefore:

- one feature pair → one 2D rotation plane
- one rotation plane → one frequency `ω`
- different pairs → different frequencies

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [19]:
from torch_judge import hint
hint("rope")


💡 Hint for Rotary Position Embedding (RoPE):
   Split into pairs (x_even, x_odd). Compute angles = pos * 1/(10000^(2i/d)). Rotate: [x_e*cos - x_o*sin, x_e*sin + x_o*cos]. Stack and flatten.



In [32]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    # pass
    assert q.size(-1) % 2 == 0
    
    positions = torch.arange(q.size(1), device=q.device).unsqueeze(1)  # (seq_len, 1)
    theta = positions / (10000 ** (torch.arange(0, q.size(-1), 2, device=q.device) / q.size(-1))) # (seq_len, d_model/2)
    
    q_even = q[..., 0::2] # even dimensions, (batch, seq_len, d_model/2)
    q_odd  = q[..., 1::2] # odd dimensions, (batch, seq_len, d_model/2)
    k_even = k[..., 0::2]
    k_odd  = k[..., 1::2]
    
    
    qr = torch.stack((q_even * torch.cos(theta) - q_odd * torch.sin(theta),
                      q_even * torch.sin(theta) + q_odd * torch.cos(theta)), dim=-1).flatten(-2) # (batch, seq_len, d_model)
    kr = torch.stack((k_even * torch.cos(theta) - k_odd * torch.sin(theta),
                      k_even * torch.sin(theta) + k_odd * torch.cos(theta)), dim=-1).flatten(-2) # (batch, seq_len, d_model)
    
    return qr, kr

In [33]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [34]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (0.6ms)
  ✅ [2/4] Preserves norm (6.4ms)
  ✅ [3/4] Relative position property (0.5ms)
  ✅ [4/4] Gradient flow (0.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.0ms total)
  Progress saved. Run status() to see your dashboard.

